In [1]:
# ============================================================
# CELL 1 — MOUNT DRIVE + CREATE NEW SIMULATION WORKSPACE
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil

# ------------------------------------------------------------
# Main research project
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ResearchProject"
)

# ------------------------------------------------------------
# Simulation-only folders
# ------------------------------------------------------------

SIM_DIR = PROJECT_ROOT / "simulation"
SCENARIO_DIR = SIM_DIR / "scenarios"
GENERATED_DIR = SIM_DIR / "generated"
OUTPUT_DIR = SIM_DIR / "outputs"

# ------------------------------------------------------------
# IMPORTANT:
# Delete ONLY the previous simulation folder.
# Research/training data are NOT touched.
# ------------------------------------------------------------

if SIM_DIR.exists():
    shutil.rmtree(SIM_DIR)

# Recreate clean simulation folders

for folder in [
    SIM_DIR,
    SCENARIO_DIR,
    GENERATED_DIR,
    OUTPUT_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 60)
print("SIMULATION WORKSPACE CREATED")
print("=" * 60)

print()
print("Project root:")
print(PROJECT_ROOT)

print()
print("Simulation directory:")
print(SIM_DIR)

print()
print("Research data remain untouched.")

ValueError: mount failed

In [2]:
# ============================================================
# CELL 2 — IMPORT LIBRARIES + EXISTING NAVIGATION MODULES
# ============================================================

import sys
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Existing Agentic AI module directory
# ------------------------------------------------------------

AGENT_DIR = PROJECT_ROOT / "agentic_ai"

if str(AGENT_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(AGENT_DIR)
    )

# ------------------------------------------------------------
# Import existing encounter classifier
# ------------------------------------------------------------

from encounter_classifier import process_frame

print("✓ Libraries loaded")
print("✓ Existing encounter classifier loaded")

print()
print("Agent directory:")
print(AGENT_DIR)

In [3]:
# ============================================================
# CELL 3 — READ THE ACTUAL ENCOUNTER DISTANCE THRESHOLD
# ============================================================

import encounter_classifier

# ------------------------------------------------------------
# Read the actual value used by your existing classifier
# ------------------------------------------------------------

SAFE_DISTANCE_KM = encounter_classifier.SAFE_DISTANCE_KM

print("=" * 60)
print("EXISTING ENCOUNTER CLASSIFIER SETTINGS")
print("=" * 60)

print()
print(
    "SAFE_DISTANCE_KM =",
    SAFE_DISTANCE_KM
)

# ------------------------------------------------------------
# Automatically choose a starting distance safely inside
# the classifier's interaction area.
# ------------------------------------------------------------

INITIAL_SEPARATION_KM = SAFE_DISTANCE_KM * 0.70

print()
print(
    "Simulation initial separation =",
    round(INITIAL_SEPARATION_KM, 6),
    "km"
)

# ------------------------------------------------------------
# Simulation settings
# ------------------------------------------------------------

NUM_FRAMES = 120
TIME_STEP_SECONDS = 10

OWN_TRACK_ID = 1
TARGET_TRACK_ID = 2

OWN_SPEED_KNOTS = 10.0
TARGET_SPEED_KNOTS = 10.0

OWN_INITIAL_COURSE = 0.0
TARGET_INITIAL_COURSE = 180.0

TURN_ANGLE_DEG = 30.0

CENTER_LAT = 33.660000
CENTER_LON = -118.282000

EARTH_RADIUS_KM = 6371.0

print()
print("✓ Simulation settings ready")

In [4]:
# ============================================================
# CELL 4 — GEOGRAPHIC MOVEMENT FUNCTIONS
# ============================================================

def knots_to_km_per_second(speed_knots):
    """
    Convert knots to kilometres per second.
    """

    return (
        speed_knots * 1.852 / 3600.0
    )


def move_vessel(
    latitude,
    longitude,
    speed_knots,
    course_deg,
    time_seconds
):
    """
    Move a vessel using:

        0°   = North
        90°  = East
        180° = South
        270° = West
    """

    distance_km = (
        knots_to_km_per_second(
            speed_knots
        )
        * time_seconds
    )

    course_rad = math.radians(
        course_deg
    )

    # North / South component
    north_km = (
        distance_km
        * math.cos(course_rad)
    )

    # East / West component
    east_km = (
        distance_km
        * math.sin(course_rad)
    )

    # Latitude change
    delta_lat = math.degrees(
        north_km / EARTH_RADIUS_KM
    )

    # Longitude change
    cos_lat = math.cos(
        math.radians(latitude)
    )

    delta_lon = math.degrees(
        east_km
        / (
            EARTH_RADIUS_KM
            * cos_lat
        )
    )

    new_latitude = (
        latitude
        + delta_lat
    )

    new_longitude = (
        longitude
        + delta_lon
    )

    return (
        new_latitude,
        new_longitude
    )


print("✓ Geographic movement functions ready")

In [5]:
# ============================================================
# CELL 5 — CREATE INITIAL HEAD-ON POSITIONS
# ============================================================

# ------------------------------------------------------------
# Half of separation
# ------------------------------------------------------------

half_separation_km = (
    INITIAL_SEPARATION_KM / 2.0
)

# Convert km to latitude degrees
half_separation_deg = math.degrees(
    half_separation_km / EARTH_RADIUS_KM
)

# ------------------------------------------------------------
# OWN SHIP
# South of center, travelling NORTH
# ------------------------------------------------------------

own_lat = (
    CENTER_LAT
    - half_separation_deg
)

own_lon = CENTER_LON

own_course = OWN_INITIAL_COURSE

# ------------------------------------------------------------
# TARGET SHIP
# North of center, travelling SOUTH
# ------------------------------------------------------------

target_lat = (
    CENTER_LAT
    + half_separation_deg
)

target_lon = CENTER_LON

target_course = TARGET_INITIAL_COURSE

print("=" * 60)
print("INITIAL HEAD-ON POSITIONS")
print("=" * 60)

print()
print(
    f"Own ship    : "
    f"{own_lat:.8f}, {own_lon:.8f}"
)

print(
    f"Target ship : "
    f"{target_lat:.8f}, {target_lon:.8f}"
)

print()
print(
    f"Initial separation : "
    f"{INITIAL_SEPARATION_KM:.6f} km"
)

print(
    f"Classifier limit   : "
    f"{SAFE_DISTANCE_KM:.6f} km"
)

print()
print(
    f"Own course         : {own_course:.1f}°"
)

print(
    f"Target course      : {target_course:.1f}°"
)

In [6]:
# ============================================================
# CELL 6 — TEST EXISTING ENCOUNTER CLASSIFIER
# ============================================================

test_frame = pd.DataFrame([

    {
        "frame": 0,
        "track_id": OWN_TRACK_ID,
        "class": "ship",
        "source": "simulation",
        "latitude": own_lat,
        "longitude": own_lon,
        "heading_deg": own_course,
        "COG": own_course,
        "SOG": OWN_SPEED_KNOTS
    },

    {
        "frame": 0,
        "track_id": TARGET_TRACK_ID,
        "class": "ship",
        "source": "simulation",
        "latitude": target_lat,
        "longitude": target_lon,
        "heading_deg": target_course,
        "COG": target_course,
        "SOG": TARGET_SPEED_KNOTS
    }
])

# ------------------------------------------------------------
# Existing project classifier
# ------------------------------------------------------------

encounter_test = process_frame(
    test_frame
)

print()
print("=" * 80)
print("ENCOUNTER RESULT")
print("=" * 80)

display(
    encounter_test
)

In [7]:
# ============================================================
# CELL 7 — CLOSED-LOOP HEAD-ON NAVIGATION SIMULATION
# ============================================================

import pandas as pd
import math

# ------------------------------------------------------------
# Reset simulation
# ------------------------------------------------------------

sim_own_lat = own_lat
sim_own_lon = own_lon

sim_target_lat = target_lat
sim_target_lon = target_lon

sim_own_course = own_course
sim_target_course = target_course

# Keep track of whether the maneuver has happened
maneuver_executed = False

# Store complete simulation history
simulation_rows = []

# ------------------------------------------------------------
# Run simulation
# ------------------------------------------------------------

for frame in range(30):

    # ========================================================
    # 1. Build current vessel state
    # ========================================================

    frame_df = pd.DataFrame([
        {
            "frame": frame,
            "track_id": OWN_TRACK_ID,
            "class": "ship",
            "source": "simulation",
            "latitude": sim_own_lat,
            "longitude": sim_own_lon,
            "heading_deg": sim_own_course,
            "COG": sim_own_course,
            "SOG": OWN_SPEED_KNOTS
        },
        {
            "frame": frame,
            "track_id": TARGET_TRACK_ID,
            "class": "ship",
            "source": "simulation",
            "latitude": sim_target_lat,
            "longitude": sim_target_lon,
            "heading_deg": sim_target_course,
            "COG": sim_target_course,
            "SOG": TARGET_SPEED_KNOTS
        }
    ])

    # ========================================================
    # 2. Run existing encounter classifier
    # ========================================================

    result = process_frame(frame_df)

    own_result = result[
        result["track_id"] == OWN_TRACK_ID
    ]

    # Default values
    distance = None
    encounter_type = "SAFE"
    risk_level = "LOW"
    risk_score = 0
    recommended_action = "MAINTAIN_COURSE"

    if len(own_result) > 0:

        row = own_result.iloc[0]

        distance = row["distance_km"]
        encounter_type = row["encounter_type"]
        risk_level = row["risk_level"]
        risk_score = row["risk_score"]
        recommended_action = row["recommended_action"]

    # ========================================================
    # 3. EXECUTE navigation decision
    # ========================================================

    action_before = recommended_action
    course_before = sim_own_course

    if recommended_action == "TURN_RIGHT":

        # Starboard/right turn
        sim_own_course = (
            sim_own_course + TURN_ANGLE_DEG
        ) % 360

        maneuver_executed = True

    elif recommended_action == "TURN_LEFT":

        # Port/left turn
        sim_own_course = (
            sim_own_course - TURN_ANGLE_DEG
        ) % 360

        maneuver_executed = True

    elif recommended_action == "SLOW_DOWN":

        # Reduce speed for this simulation
        # We keep the course unchanged.
        pass

    elif recommended_action == "STOP":

        # For now, stop own vessel
        pass

    # ========================================================
    # 4. Save state BEFORE movement
    # ========================================================

    simulation_rows.append({
        "frame": frame,

        "own_lat": sim_own_lat,
        "own_lon": sim_own_lon,

        "target_lat": sim_target_lat,
        "target_lon": sim_target_lon,

        "own_course_before": course_before,
        "own_course_after": sim_own_course,

        "target_course": sim_target_course,

        "distance_km": distance,

        "encounter_type": encounter_type,
        "risk_level": risk_level,
        "risk_score": risk_score,

        "recommended_action": recommended_action,

        "maneuver_executed": (
            sim_own_course != course_before
        )
    })

    # ========================================================
    # 5. Move OWN SHIP using NEW course
    # ========================================================

    sim_own_lat, sim_own_lon = move_vessel(
        sim_own_lat,
        sim_own_lon,
        OWN_SPEED_KNOTS,
        sim_own_course,
        TIME_STEP_SECONDS
    )

    # ========================================================
    # 6. Move TARGET SHIP
    # ========================================================

    sim_target_lat, sim_target_lon = move_vessel(
        sim_target_lat,
        sim_target_lon,
        TARGET_SPEED_KNOTS,
        sim_target_course,
        TIME_STEP_SECONDS
    )


# ============================================================
# Create DataFrame
# ============================================================

closed_loop_df = pd.DataFrame(
    simulation_rows
)

print("=" * 100)
print("CLOSED-LOOP HEAD-ON NAVIGATION SIMULATION")
print("=" * 100)

display(
    closed_loop_df[
        [
            "frame",
            "distance_km",
            "encounter_type",
            "risk_level",
            "risk_score",
            "recommended_action",
            "own_course_before",
            "own_course_after",
            "maneuver_executed"
        ]
    ]
)

In [8]:
# ============================================================
# CELL 8 — VERIFY THAT TURN RIGHT ACTUALLY CHANGED TRAJECTORY
# ============================================================

maneuvers = closed_loop_df[
    closed_loop_df["maneuver_executed"] == True
]

print("=" * 80)
print("EXECUTED MANEUVERS")
print("=" * 80)

display(
    maneuvers[
        [
            "frame",
            "distance_km",
            "encounter_type",
            "risk_level",
            "recommended_action",
            "own_course_before",
            "own_course_after"
        ]
    ]
)

print("\nFinal own-ship course:",
      closed_loop_df.iloc[-1]["own_course_after"])

print(
    "\nNumber of executed maneuvers:",
    len(maneuvers)
)

In [9]:
# ============================================================
# GRAPHICAL HEAD-ON SIMULATION — LIGHTER WATER + DARK NAVIGATION
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Polygon, FancyBboxPatch
from IPython.display import HTML, display

# ------------------------------------------------------------
# SIMULATION SETTINGS
# ------------------------------------------------------------

FPS = 20
DURATION = 18
N_FRAMES = FPS * DURATION

OWN_SPEED = 0.055
TARGET_SPEED = 0.055

OWN_START = np.array([0.0, -8.0])
TARGET_START = np.array([0.0, 8.0])

INITIAL_OWN_COURSE = 0.0
TARGET_COURSE = 180.0

# Turn earlier, at a safe distance
TURN_TRIGGER_DISTANCE = 5.0
TURN_ANGLE = 30.0

# ------------------------------------------------------------
# CREATE TRAJECTORY
# ------------------------------------------------------------

own_positions = []
target_positions = []
own_courses = []
distances = []
actions = []

own_pos = OWN_START.copy()
target_pos = TARGET_START.copy()

own_course = INITIAL_OWN_COURSE
turn_done = False

for frame in range(N_FRAMES):

    distance = np.linalg.norm(target_pos - own_pos)

    # Navigation decision
    if not turn_done and distance <= TURN_TRIGGER_DISTANCE:
        action = "TURN RIGHT"
        own_course = TURN_ANGLE
        turn_done = True
    elif turn_done:
        action = "TURN RIGHT — EXECUTING"
    else:
        action = "MAINTAIN COURSE"

    own_positions.append(own_pos.copy())
    target_positions.append(target_pos.copy())
    own_courses.append(own_course)
    distances.append(distance)
    actions.append(action)

    # Convert course to movement
    own_rad = np.radians(own_course)
    target_rad = np.radians(TARGET_COURSE)

    own_velocity = np.array([
        np.sin(own_rad),
        np.cos(own_rad)
    ]) * OWN_SPEED

    target_velocity = np.array([
        np.sin(target_rad),
        np.cos(target_rad)
    ]) * TARGET_SPEED

    own_pos = own_pos + own_velocity
    target_pos = target_pos + target_velocity


own_positions = np.array(own_positions)
target_positions = np.array(target_positions)

# ------------------------------------------------------------
# FIGURE
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 11))

# LIGHTER WATER
ax.set_facecolor("#9fd8e8")
fig.patch.set_facecolor("#dff4f8")

ax.set_xlim(-5, 5)
ax.set_ylim(-9, 9)

ax.set_aspect("equal")

ax.set_title(
    "HEAD-ON MARITIME ENCOUNTER — NAVIGATION SIMULATION",
    fontsize=15,
    fontweight="bold",
    color="#102a43",
    pad=15
)

ax.set_xlabel(
    "Cross-track distance",
    fontsize=11,
    color="#16324f"
)

ax.set_ylabel(
    "Along-track distance",
    fontsize=11,
    color="#16324f"
)

# ------------------------------------------------------------
# WATER GRID / CENTER LINE
# ------------------------------------------------------------

ax.axvline(
    0,
    linestyle="--",
    linewidth=1.5,
    alpha=0.45,
    color="#477b8c"
)

ax.grid(
    True,
    alpha=0.20,
    color="#ffffff"
)

# ------------------------------------------------------------
# SHIP GRAPHICS
# ------------------------------------------------------------

own_ship = Polygon(
    [
        [0, 0.55],
        [-0.28, -0.40],
        [0.28, -0.40]
    ],
    closed=True,
    facecolor="#0b3d91",
    edgecolor="#061f4a",
    linewidth=2
)

target_ship = Polygon(
    [
        [0, -0.55],
        [-0.28, 0.40],
        [0.28, 0.40]
    ],
    closed=True,
    facecolor="#c62828",
    edgecolor="#651414",
    linewidth=2
)

ax.add_patch(own_ship)
ax.add_patch(target_ship)

# ------------------------------------------------------------
# TRAILS
# ------------------------------------------------------------

own_trail, = ax.plot(
    [],
    [],
    linewidth=2,
    color="#083b66",
    alpha=0.75
)

target_trail, = ax.plot(
    [],
    [],
    linewidth=2,
    color="#7f1d1d",
    alpha=0.70
)

# ------------------------------------------------------------
# SHIP LABELS
# ------------------------------------------------------------

own_label = ax.text(
    0,
    -8.65,
    "OWN VESSEL",
    ha="center",
    fontsize=11,
    fontweight="bold",
    color="#062a5c"
)

target_label = ax.text(
    0,
    8.55,
    "TARGET VESSEL",
    ha="center",
    fontsize=11,
    fontweight="bold",
    color="#7f1515"
)

# ------------------------------------------------------------
# NAVIGATION DECISION PANEL
# ------------------------------------------------------------

nav_box = FancyBboxPatch(
    (-4.65, 5.1),
    9.3,
    3.15,
    boxstyle="round,pad=0.15",
    linewidth=2,
    facecolor="#f8fbfc",
    edgecolor="#17324d",
    alpha=0.96
)

ax.add_patch(nav_box)

nav_text = ax.text(
    -4.35,
    7.95,
    "",
    fontsize=11.5,
    verticalalignment="top",
    color="#071b2c",       # DARK TEXT
    fontweight="bold",
    linespacing=1.45
)

# ------------------------------------------------------------
# DISTANCE TEXT
# ------------------------------------------------------------

distance_text = ax.text(
    0,
    -7.8,
    "",
    ha="center",
    fontsize=12,
    fontweight="bold",
    color="#071b2c"
)

# ------------------------------------------------------------
# UPDATE FUNCTION
# ------------------------------------------------------------

def update(frame):

    own = own_positions[frame]
    target = target_positions[frame]

    distance = distances[frame]
    course = own_courses[frame]
    action = actions[frame]

    # --------------------------------------------------------
    # OWN SHIP
    # --------------------------------------------------------

    angle = np.radians(course)

    forward = np.array([
        np.sin(angle),
        np.cos(angle)
    ])

    right = np.array([
        np.cos(angle),
        -np.sin(angle)
    ])

    nose = own + forward * 0.55
    rear = own - forward * 0.40

    left_point = rear - right * 0.28
    right_point = rear + right * 0.28

    own_ship.set_xy([
        nose,
        left_point,
        right_point
    ])

    # --------------------------------------------------------
    # TARGET SHIP
    # --------------------------------------------------------

    target_angle = np.radians(TARGET_COURSE)

    target_forward = np.array([
        np.sin(target_angle),
        np.cos(target_angle)
    ])

    target_right = np.array([
        np.cos(target_angle),
        -np.sin(target_angle)
    ])

    target_nose = target + target_forward * 0.55
    target_rear = target - target_forward * 0.40

    target_left = target_rear - target_right * 0.28
    target_right_point = target_rear + target_right * 0.28

    target_ship.set_xy([
        target_nose,
        target_left,
        target_right_point
    ])

    # --------------------------------------------------------
    # TRAILS
    # --------------------------------------------------------

    own_trail.set_data(
        own_positions[:frame + 1, 0],
        own_positions[:frame + 1, 1]
    )

    target_trail.set_data(
        target_positions[:frame + 1, 0],
        target_positions[:frame + 1, 1]
    )

    # --------------------------------------------------------
    # NAVIGATION DECISION
    # --------------------------------------------------------

    if action == "MAINTAIN COURSE":

        decision = "MAINTAIN COURSE"
        decision_status = "MONITORING"

    elif action == "TURN RIGHT":

        decision = "TURN RIGHT"
        decision_status = "DECISION TRIGGERED"

    else:

        decision = "TURN RIGHT"
        decision_status = "MANEUVER EXECUTING"

    nav_text.set_text(
        f"FRAME                 : {frame:03d}\n"
        f"ENCOUNTER             : HEAD-ON\n"
        f"RISK DISTANCE         : {distance:.2f}\n"
        f"OWN COURSE            : {course:05.1f}°\n"
        f"TARGET COURSE         : 180.0°\n"
        f"NAVIGATION DECISION   : {decision}\n"
        f"STATUS                : {decision_status}"
    )

    distance_text.set_text(
        f"SEPARATION: {distance:.2f} units"
    )

    return (
        own_ship,
        target_ship,
        own_trail,
        target_trail,
        nav_text,
        distance_text
    )


# ------------------------------------------------------------
# ANIMATION
# ------------------------------------------------------------

anim = FuncAnimation(
    fig,
    update,
    frames=N_FRAMES,
    interval=1000 / FPS,
    blit=False
)

plt.close(fig)

display(HTML(anim.to_jshtml()))